# RL Dynamic Pricing - Exploration Notebook

This notebook is for exploring the pricing environment and testing components.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path().absolute().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from environment.pricing_env import PricingEnv
from config.constants import PRICE_LEVELS, N_PRICE_ACTIONS

%matplotlib inline
sns.set_style('whitegrid')

## 1. Environment Testing

In [ ]:
# Create environment
env = PricingEnv(seed=42)

# Reset and run one episode
obs, info = env.reset(seed=42)
print(f"Initial observation: {obs}")
print(f"Initial info: {info}")

In [ ]:
# Run a random policy episode
obs, info = env.reset(seed=42)
total_reward = 0

for step in range(30):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    
    if terminated:
        break

print(f"Episode finished!")
print(f"Total reward: {total_reward:.4f}")
print(f"Total profit: ${info['total_profit']:.2f}")

## 2. Visualize Episode

In [ ]:
# Get history DataFrame
history_df = env.get_history_df()
history_df.head(10)

In [ ]:
# Plot episode
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Price over time
axes[0].plot(history_df['day'], history_df['prices'], marker='o')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Price Over Time')
axes[0].grid(True)

# Demand vs Sales
axes[1].bar(history_df['day'], history_df['demands'], alpha=0.5, label='Demand')
axes[1].plot(history_df['day'], history_df['sales'], marker='o', color='red', label='Sales')
axes[1].set_ylabel('Units')
axes[1].set_title('Demand vs Sales')
axes[1].legend()
axes[1].grid(True)

# Inventory
axes[2].fill_between(history_df['day'], history_df['inventory'], alpha=0.3)
axes[2].plot(history_df['day'], history_df['inventory'], marker='o')
axes[2].set_xlabel('Day')
axes[2].set_ylabel('Inventory')
axes[2].set_title('Inventory Level')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 3. Demand Analysis

In [ ]:
# Test demand at different prices
test_prices = PRICE_LEVELS
demands = []

env = PricingEnv(seed=42)

for price in test_prices:
    # Reset with specific price
    price_idx = PRICE_LEVELS.index(price)
    obs, _ = env.reset(seed=42, options={'initial_price_idx': price_idx})
    
    # Get demand for this price
    demand = env._compute_demand(price)
    demands.append(demand)

plt.figure(figsize=(10, 6))
plt.plot(test_prices, demands, marker='o')
plt.xlabel('Price ($)')
plt.ylabel('Demand')
plt.title('Price-Demand Relationship')
plt.grid(True)
plt.show()

## 4. Multiple Episodes Comparison

In [ ]:
# Run multiple episodes with different strategies
n_episodes = 10
results = {
    'random': [],
    'fixed_low': [],
    'fixed_mid': [],
    'fixed_high': []
}

for i in range(n_episodes):
    # Random strategy
    env = PricingEnv(seed=i)
    obs, _ = env.reset(seed=i)
    done = False
    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
    results['random'].append(info['total_profit'])
    
    # Fixed low price
    env = PricingEnv(seed=i)
    obs, _ = env.reset(seed=i, options={'initial_price_idx': 0})
    done = False
    while not done:
        obs, reward, terminated, truncated, info = env.step(0)
        done = terminated or truncated
    results['fixed_low'].append(info['total_profit'])
    
    # Fixed mid price
    mid_idx = N_PRICE_ACTIONS // 2
    env = PricingEnv(seed=i)
    obs, _ = env.reset(seed=i, options={'initial_price_idx': mid_idx})
    done = False
    while not done:
        obs, reward, terminated, truncated, info = env.step(mid_idx)
        done = terminated or truncated
    results['fixed_mid'].append(info['total_profit'])
    
    # Fixed high price
    env = PricingEnv(seed=i)
    obs, _ = env.reset(seed=i, options={'initial_price_idx': N_PRICE_ACTIONS-1})
    done = False
    while not done:
        obs, reward, terminated, truncated, info = env.step(N_PRICE_ACTIONS-1)
        done = terminated or truncated
    results['fixed_high'].append(info['total_profit'])

# Plot comparison
plt.figure(figsize=(10, 6))
for strategy, profits in results.items():
    plt.plot(range(n_episodes), profits, marker='o', label=strategy)
plt.xlabel('Episode')
plt.ylabel('Total Profit ($)')
plt.title('Strategy Comparison')
plt.legend()
plt.grid(True)
plt.show()

# Print summary
for strategy, profits in results.items():
    print(f"{strategy}: mean=${np.mean(profits):.2f}, std=${np.std(profits):.2f}")

## 5. Load and Explore Data

In [ ]:
# Load synthetic data if available
data_path = Path('../data/synthetic_sales.csv')

if data_path.exists():
    df = pd.read_csv(data_path, parse_dates=['date'])
    print(f"Loaded {len(df)} records")
    print(df.head())
    
    # Visualize
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Price distribution
    axes[0, 0].hist(df['price'], bins=30, edgecolor='black')
    axes[0, 0].set_title('Price Distribution')
    axes[0, 0].set_xlabel('Price ($)')
    
    # Demand distribution
    axes[0, 1].hist(df['demand'], bins=30, edgecolor='black')
    axes[0, 1].set_title('Demand Distribution')
    axes[0, 1].set_xlabel('Demand')
    
    # Price vs Demand scatter
    axes[1, 0].scatter(df['price'], df['demand'], alpha=0.3)
    axes[1, 0].set_xlabel('Price ($)')
    axes[1, 0].set_ylabel('Demand')
    axes[1, 0].set_title('Price vs Demand')
    
    # Revenue over time
    df_monthly = df.set_index('date').resample('M')['revenue'].sum()
    axes[1, 1].plot(df_monthly.index, df_monthly.values, marker='o')
    axes[1, 1].set_title('Monthly Revenue')
    axes[1, 1].set_ylabel('Revenue ($)')
    
    plt.tight_layout()
    plt.show()
else:
    print("No data file found. Generate data first:")
    print("python -m data.generate_data")

## Next Steps

1. Train the demand predictor: `python -m training.train_demand_predictor`
2. Train the PPO agent: `python -m training.train_ppo`
3. Launch dashboard: `streamlit run dashboard/streamlit_app.py`